In [1]:
import torch 
from eb_jepa.datasets.moving_mnist import MovingMNISTDet
from architectures import ResNet5, StateOnlyPredictor, ResUNet

In [2]:
train_data = MovingMNISTDet(split="train")

In [3]:
single_video = train_data[0]["video"]
print(single_video.shape)

torch.Size([1, 10, 64, 64])


In [4]:
D_OBS = 1  # input channels (gray scale) 
H_ENC = 32 # hidden dim in encoder 
D_STC = 16 # represenation dim (encoder output channels) 
H_PRE = 32   # hidden dim in predictor

observations = torch.randn(32,1,10,64,64)
encoder = ResNet5(in_d=D_OBS, h_d=H_ENC, out_d= D_STC)
with torch.no_grad(): 
    state = encoder(observations)
    print(state.shape)

torch.Size([32, 16, 10, 64, 64])


In [5]:
predictor_net = ResUNet(2 * D_STC, H_PRE, D_STC)        # input is 2x D_STC (two frames concatenated)
predictor     = StateOnlyPredictor(predictor_net, context_length=2)

In [6]:
return_all_steps = True 
all_steps = []
context_length = getattr(predictor, "context_length")

nsteps = 4 
action_encoded = None 

predicted_states = state 
for _ in range(nsteps): 
    with torch.no_grad(): 
        predicted_states = predictor(predicted_states,action_encoded)[:, :, :-1]
    print(predicted_states.shape)

    if return_all_steps: 
        all_steps.append(predicted_states)

    predicted_states = torch.cat((state[:, :, :context_length], predicted_states), dim=2)
    


torch.Size([32, 16, 8, 64, 64])
torch.Size([32, 16, 8, 64, 64])
torch.Size([32, 16, 8, 64, 64])
torch.Size([32, 16, 8, 64, 64])


In [7]:
print(len(all_steps))

4


In [8]:
all_steps[0].shape

torch.Size([32, 16, 8, 64, 64])